# Scénario 11b : anticiper la codification de septembre, cible binaire (sain ou en retard)

Journal : [`tableau_resultats.md`](tableau_resultats.md). Essai **bonus et indépendant** des autres branches de machine learning du projet.

**Objet** : prédire **uniquement `PAY_1`**, la codification de septembre, à partir de son **historique** (avril à septembre), en **deux classes** : **sain (-2, -1 ou 0)** ou **en retard (2 et plus)**, sur les clients dont `PAY_1` n'est pas un 1 ; puis **enregistrer les cinq modèles réglés** dans `modeles/11b/`. La simplification en deux classes allège le travail des modèles sur la question qui compte (sortir du retard, y rester, y entrer) ; la codification exacte (-2, -1 ou 0) n'est plus prédite. Les modèles serviront ensuite à estimer ce que cachent les `PAY_1` = 1.

**Données** : `data/creditcard_pret_ingestion.csv` (niveau 0), **sans `dpnm`, sans `ID`, sans données personnelles**, **sans correctifs** (codifications d'origine) ; un scénario avec les correctifs (`10`, puis la règle fixe des retards) viendra ensuite, pour comparer. **Pas d'empilement des mois** : un exemple par client, septembre seulement.

**Méthode** : celle du scénario `01e` (septembre seul, montants et plafond au logarithme signé, codifications d'avril à août regroupées en -2, -1, 0, 2 et plus), avec les ajouts devenus la règle de l'essai : boucle d'optimisation partant des **meilleurs réglages du `01e`**, une ligne par tour, régression logistique avec choix de pénalité L1, profondeur des arbres plafonnée à 30, surapprentissage limité à 7 % relatif, alerte si l'écart-type des plis dépasse 2 points, importance des variables. Validation croisée à 5 plis sur le train (21 049 clients) ; le test de septembre n'est pas lu. Pas de réapprentissage supplémentaire : chaque modèle enregistré est appris sur tout le train avec son meilleur réglage.

## 1. Données

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
print(f"Clients du niveau 0 : {len(df)} ; colonnes : {df.shape[1]} (dpnm retirée) ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())}")

# Codifications 1 d'avril à août : leurs clients ont tous PAY_1 = 1, ils restent donc hors du train et du test
anciens_1 = (df[PAY[1:]] == 1).any(axis=1)
assert (df.loc[anciens_1, "PAY_1"] == 1).all()
print(f"Clients avec une codification 1 d'avril à août : {int(anciens_1.sum())}, tous codifiés 1 aussi en septembre")

Clients du niveau 0 : 30000 ; colonnes : 24 (dpnm retirée) ; PAY_1 = 1 : 3688
Clients avec une codification 1 d'avril à août : 28, tous codifiés 1 aussi en septembre


## 2. Septembre : ce que la banque voyait, et ce qu'on cherche à retrouver

**Cible** : la codification de `PAY_1`, **telle quelle**, de -2 à 8 (une classe par codification, sans regrouper les retards) : le niveau du retard fait partie de ce que le modèle doit comprendre. Un essai avec les retards regroupés viendra après, seulement si le résultat est peu probant.

**Variables** : les colonnes du dataset d'origine (niveau 0), sauf `ID` (un identifiant), `PAY_1` (la cible), `dpnm` et les données personnelles ; **aucune colonne ajoutée** :
- les codifications d'avril à août (`PAY_2` à `PAY_6`), telles quelles : les 28 clients qui y ont une codification 1 l'ont tous aussi en septembre, ils ne sont donc ni dans le train ni dans le test ;
- les soldes de fin de mois (`BILL_AMT1` à `BILL_AMT6`), les paiements (`PAY_AMT1` à `PAY_AMT6`) et le plafond (`LIMIT_BAL`).

Les données personnelles (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`) sont retirées : elles ne sont normalement pas pertinentes pour une codification, qui est un statut de paiement.

Ce ML est un **essai indépendant** : il ne reprend aucun choix des autres branches de ML du projet.

**Exemples** : les clients dont `PAY_1` n'est pas un 1, coupés en 80/20 (train / test, `stratify` sur la classe, `random_state=42`). Les `PAY_1` = 1 sont mis de côté : ce sont eux qu'on cherche à trancher.

In [2]:
from sklearn.model_selection import train_test_split

# Variables : colonnes du dataset d'origine (niveau 0), sauf ID (identifiant), PAY_1 (la cible), dpnm et les données personnelles ; aucune colonne ajoutée
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
NUMERIQUES = ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
CATEGORIELLES = PAY[1:]
VARIABLES = CATEGORIELLES + NUMERIQUES
assert sorted(VARIABLES + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)
mask_1 = df["PAY_1"] == 1
connus = df[~mask_1]
a_trancher = df[mask_1]

X_train, X_test, y_train, y_test = train_test_split(
    connus[VARIABLES], connus["PAY_1"], test_size=0.2, stratify=connus["PAY_1"], random_state=42)
CLASSES = sorted(connus["PAY_1"].unique())   # toutes les codifications de PAY_1, sauf 1

repartition = pd.DataFrame({
    "Train": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
repartition.loc["Total"] = repartition.sum()
print(f"PAY_1 connus : {len(connus)} (train {len(X_train)}, test {len(X_test)}) ; PAY_1 = 1 à trancher : {len(a_trancher)}")
print(f"Variables ({len(VARIABLES)}) : {VARIABLES}")
display(repartition)

PAY_1 connus : 26312 (train 21049, test 5263) ; PAY_1 = 1 à trancher : 3688
Variables (18) : ['PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'LIMIT_BAL', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


,Train,Test
PAY_1,,
-2,2207,552
-1,4549,1137
0,11789,2948
2,2133,534
3,258,64
4,61,15
5,21,5
6,9,2
7,7,2


**Comment lire** : nombre de clients par codification de `PAY_1`, dans le train et dans le test. Les proportions sont les mêmes dans les deux (`stratify`). Le test n'est plus touché jusqu'au critère A.

## 3. Les niveaux de retard d'août à septembre (train)

`PAY_2` compte des retards importants (3 à 8) qui repassent à 2 en septembre, ce qui peut perturber les modèles. Le tableau croise, sur les clients du train, la codification d'août (en ligne) et celle de septembre (en colonne) : les codifications saines (-2, -1, 0) sont réunies, les niveaux de retard sont détaillés.

In [3]:
def libelle_retard(codifications):
    # Niveaux de retard détaillés, codifications saines réunies
    return codifications.astype(int).astype(str).where(codifications >= 2, "sain (-2, -1, 0)")

croisement = pd.crosstab(libelle_retard(X_train["PAY_2"]).rename("Août (PAY_2)"),
                         libelle_retard(y_train).rename("Septembre (PAY_1)"), margins=True, margins_name="Total")
display(croisement)

Septembre (PAY_1),2,3,4,5,6,7,8,"sain (-2, -1, 0)",Total
Août (PAY_2),,,,,,,,,
2,1266,216,0,0,0,0,0,326,1808
3,57,35,46,0,0,0,0,35,173
4,11,6,12,20,0,0,0,4,53
5,0,1,3,0,9,0,0,1,14
6,0,0,0,1,0,7,0,0,8
7,0,0,0,0,0,0,15,0,15
"sain (-2, -1, 0)",799,0,0,0,0,0,0,18179,18978
Total,2133,258,61,21,9,7,15,18545,21049


**Comment lire** : chaque case donne le nombre de clients du train passés de la codification d'août (ligne) à celle de septembre (colonne). La diagonale correspond aux retards gardés au même niveau ; sous la diagonale, à gauche, les retards qui baissent (par exemple de 3 en août à 2 en septembre) ; au-dessus, ceux qui montent. La ligne et la colonne « Total » donnent le nombre de codifications de chaque niveau, en août et en septembre.

## 3 bis. Regroupement des retards

La cible `PAY_1` est **binaire** : **0** pour sain (-2, -1 ou 0), **2** pour en retard (2 à 8). Les variables `PAY_2` à `PAY_6` sont **regroupées de la même façon**.

Le regroupement se fait en mémoire, sur le train et le test ; le fichier d'origine n'est pas modifié. Le tableau précédent (niveaux de retard d'août à septembre) est calculé avant, sur les codifications détaillées.

In [4]:
def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 inchangés
    return codifications.where(codifications < 2, 2)

y_train, y_test = regrouper(y_train), regrouper(y_test)
y_train, y_test = y_train.where(y_train >= 2, 0), y_test.where(y_test >= 2, 0)   # 11b : sain -> 0, retard -> 2
for col in PAY[1:]:
    X_train[col], X_test[col] = regrouper(X_train[col]), regrouper(X_test[col])
CLASSES = sorted(y_train.unique())
print(f"Cible PAY_1 binaire : classes {CLASSES} (0 = sain, 2 = en retard)")
print(y_train.value_counts().reindex(CLASSES).rename("Train").to_frame().join(y_test.value_counts().rename("Test")))
print("Variables PAY_2 à PAY_6 regroupées : valeurs", sorted(pd.unique(X_train[PAY[1:]].values.ravel())))

Cible PAY_1 binaire : classes [np.int64(0), np.int64(2)] (0 = sain, 2 = en retard)
       Train  Test
PAY_1             
0      18545  4637
2       2504   626
Variables PAY_2 à PAY_6 regroupées : valeurs [np.int64(-2), np.int64(-1), np.int64(0), np.int64(2)]


## 4. Le score optimisé : la moyenne des taux de bonne classe (sain, en retard)

Chaque classe pèse autant : la moyenne du taux de bonnes prédictions parmi les sains et parmi les retards (`balanced_accuracy`). Le pourcentage de bonnes prédictions global reste affiché dans les tableaux de la section 8.

In [5]:
# ==============================================================================
# Score optimisé (choix validé) : "familles équilibrées" ; autre possibilité : "bonnes prédictions"
# ==============================================================================
SCORE = "familles équilibrées"
SURAPPRENTISSAGE_MAX = 7.0   # écart relatif train - validation, en %
VARIANCE_MAX = 2.0           # écart-type des 5 plis, en points
MAX_TOURS = 10               # tours de la boucle, par modèle

import time
from sklearn.base import clone
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FAMILLES = [0, 2]   # 11b : deux classes
NOMS_FAMILLES = {0: "sain (-2, -1, 0)", 2: "2 et plus"}

def famille_pay(codifications):
    # Codification -> famille : -2, -1, 0 gardés, 2 et plus réunis en 2
    v = np.asarray(codifications).ravel().astype(int)
    return np.where(v >= 2, 2, 0)   # 11b : sain (-2, -1, 0) -> 0, retard -> 2

def score_familles(vrai, predit):
    v, p = famille_pay(vrai), famille_pay(predit)
    return balanced_accuracy_score(v, p) if SCORE == "familles équilibrées" else (v == p).mean()

assert SCORE in ("familles équilibrées", "bonnes prédictions")
scoreur = make_scorer(score_familles)

def bonnes_predictions(vrai, predit):
    # Pourcentage de bonnes prédictions (bonne famille), en tout et dans chaque famille
    v, p = famille_pay(vrai), famille_pay(predit)
    resultat = {"global": (v == p).mean() * 100}
    for f in FAMILLES:
        resultat[f] = (p[v == f] == f).mean() * 100
    return resultat

# Logarithme signé des montants dus, payés et du plafond, comme dans 01b
MONTANTS_DUS_PAYES = [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)] + ["LIMIT_BAL"]

def log_montants(X):
    X = X.copy()
    X[MONTANTS_DUS_PAYES] = np.sign(X[MONTANTS_DUS_PAYES]) * np.log1p(np.abs(X[MONTANTS_DUS_PAYES]))
    return X

def etape_log():
    return FunctionTransformer(log_montants)

def preprocesseur_lineaire():
    # Codifications : une colonne par valeur ; nombres (déjà au logarithme) : mise à l'échelle standard
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", StandardScaler(), NUMERIQUES),
    ])

def preprocesseur_arbres():
    # Catégories : une colonne par valeur ; nombres tels quels (déjà au logarithme)
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Score optimisé : {SCORE} ; surapprentissage max {SURAPPRENTISSAGE_MAX} % relatif ; alerte si écart-type des plis > {VARIANCE_MAX} points")

Score optimisé : familles équilibrées ; surapprentissage max 7.0 % relatif ; alerte si écart-type des plis > 2.0 points


## 5. Les réglages explorés

Pour chaque modèle : les **échelles** (paramètres ordonnés, explorés par la boucle autour d'une valeur de départ : ici, **le meilleur réglage du 01e**) et les **choix** (paramètres sans ordre, tous essayés à chaque tour). Les autres paramètres gardent leur valeur par défaut, sauf ceux indiqués en commentaire.

- **Régression logistique** : force de la régularisation `C` (petit = forte régularisation) ; poids des classes ; **type de pénalité** (`l1_ratio` : 0 = L2, 1 = L1, qui peut annuler des variables, 0,5 = mélange), solveur `saga`.
- **Arbre de décision** : profondeur maximale (plafonnée à 30), taille minimale des feuilles (grande = règles plus générales) ; poids des classes ; critère de coupe (`gini` ou `entropy`).
- **RandomForest** : nombre d'arbres, profondeur maximale (plafonnée à 30), taille minimale des feuilles ; part des variables essayées à chaque coupe (`max_features`) ; poids des classes.
- **MLP** : taille des couches cachées, régularisation `alpha` ; arrêt anticipé (`early_stopping`) pour éviter d'apprendre par cœur.
- **CatBoost** : réglé comme dans le machine learning de la deuxième itération (même pipeline que les arbres, codifications en une colonne par valeur ; tous les cœurs ; 1 000 itérations). La boucle explore la profondeur, le `learning_rate` et `l2_leaf_reg` ; poids des classes : `Balanced` ou `SqrtBalanced` (plus doux), toujours actifs, CatBoost refusant la valeur « aucun » passée explicitement.

In [6]:
ESPACES = {
    "Régression logistique": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), LogisticRegression(solver="saga", max_iter=5000)),   # saga : accepte la pénalité L1
        "n_jobs": -1,
        "echelles": {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]},
        "depart": {"logisticregression__C": 10},   # meilleur réglage du 01e
        "choix": {"logisticregression__class_weight": [None, "balanced"],
                  "logisticregression__l1_ratio": [0, 0.5, 1]},   # 0 : L2 ; 1 : L1 (peut annuler des variables) ; 0,5 : mélange
    },
    "Arbre de décision": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), DecisionTreeClassifier(random_state=42)),
        "n_jobs": -1,
        "echelles": {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10, 12, 15, 20, 30],   # plafond à 30
                     "decisiontreeclassifier__min_samples_leaf": [1, 2, 5, 10, 20, 50, 100, 200]},
        "depart": {"decisiontreeclassifier__max_depth": 8, "decisiontreeclassifier__min_samples_leaf": 2},   # meilleur réglage du 01e
        "choix": {"decisiontreeclassifier__class_weight": [None, "balanced"],
                  "decisiontreeclassifier__criterion": ["gini", "entropy"]},
    },
    "RandomForest": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), RandomForestClassifier(random_state=42, n_jobs=-1)),
        "n_jobs": -1,
        "echelles": {"randomforestclassifier__n_estimators": [100, 200, 300, 500, 1000],
                     "randomforestclassifier__max_depth": [6, 8, 10, 12, 15, 20, 25, 30],   # plafond à 30
                     "randomforestclassifier__min_samples_leaf": [1, 2, 3, 5, 10, 20, 50]},
        "depart": {"randomforestclassifier__n_estimators": 500, "randomforestclassifier__max_depth": 25, "randomforestclassifier__min_samples_leaf": 5},   # meilleur réglage du 01e
        "choix": {"randomforestclassifier__max_features": ["sqrt", 0.5],
                  "randomforestclassifier__class_weight": [None, "balanced_subsample"]},
    },
    "MLP": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), MLPClassifier(max_iter=500, early_stopping=True, random_state=42)),
        "n_jobs": -1,
        "echelles": {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (100,), (128, 64), (256, 128), (256, 128, 64)],
                     "mlpclassifier__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
        "depart": {"mlpclassifier__hidden_layer_sizes": (128, 64), "mlpclassifier__alpha": 1e-4},   # meilleur réglage du 01e
        "choix": {},
    },
    "CatBoost": {
        # Réglé comme dans le machine learning de la deuxième itération : même pipeline que les arbres (codifications en une colonne par valeur),
        # tous les cœurs, pas de fichiers écrits ; poids des classes toujours actifs (CatBoost refuse auto_class_weights=None passé explicitement)
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(),
                                CatBoostClassifier(loss_function="MultiClass", iterations=1000, random_seed=42, verbose=0,
                                                   thread_count=-1, allow_writing_files=False)),
        "n_jobs": -1,
        "echelles": {"catboostclassifier__depth": [2, 3, 4, 5, 6, 7, 8, 10],
                     "catboostclassifier__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1, 0.2],
                     "catboostclassifier__l2_leaf_reg": [1, 3, 5, 10, 15, 30, 60]},
        "depart": {"catboostclassifier__depth": 5, "catboostclassifier__learning_rate": 0.03, "catboostclassifier__l2_leaf_reg": 5},   # meilleur réglage du 01e
        "choix": {"catboostclassifier__auto_class_weights": ["Balanced", "SqrtBalanced"]},
    },
}
for nom, e in ESPACES.items():
    for p, v in e["depart"].items():
        assert v in e["echelles"][p], (nom, p)
print("Modèles :", list(ESPACES))

Modèles : ['Régression logistique', 'Arbre de décision', 'RandomForest', 'MLP', 'CatBoost']


## 6. La boucle d'optimisation

Pour chaque modèle, tour après tour : GridSearch en validation croisée (5 plis) sur la grille courante, puis choix du meilleur réglage **parmi ceux qui surapprennent de 7 % relatif au plus**. Pour chaque modèle, une ligne : son meilleur réglage admis, son score en validation croisée et la raison de l'arrêt de la boucle.

In [7]:
def fenetre(echelle, position):
    # La valeur courante et ses deux voisines dans l'échelle
    return echelle[max(0, position - 1): position + 2]

def libelle(params):
    return ", ".join(f"{k.split('__')[1]}={v}" for k, v in params.items())

def optimiser(nom, espace):
    positions = {p: espace["echelles"][p].index(v) for p, v in espace["depart"].items()}
    lignes, raison, debut = [], f"{MAX_TOURS} tours atteints", time.perf_counter()
    for tour in range(1, MAX_TOURS + 1):
        debut_tour = time.perf_counter()
        grille = {p: fenetre(espace["echelles"][p], i) for p, i in positions.items()} | espace["choix"]
        recherche = GridSearchCV(espace["modele"], grille, scoring=scoreur, cv=cv, n_jobs=espace["n_jobs"], return_train_score=True, refit=False)
        recherche.fit(X_train, y_train, **espace.get("fit_params", {}))
        r = recherche.cv_results_
        res = pd.DataFrame({
            "tour": tour,
            "params": r["params"],
            "validation": r["mean_test_score"] * 100,
            "écart-type des plis": r["std_test_score"] * 100,
            "train": r["mean_train_score"] * 100,
        })
        res["surapprentissage (% relatif)"] = (res["train"] - res["validation"]) / res["train"] * 100
        res["admis"] = res["surapprentissage (% relatif)"] <= SURAPPRENTISSAGE_MAX
        lignes.append(res)
        admis = res[res["admis"]]
        if admis.empty:
            print(f"{nom}, tour {tour} : {len(res)} réglages, aucun admis ({round(time.perf_counter() - debut_tour)} s)", flush=True)
            raison = f"tour {tour} : aucun réglage sous {SURAPPRENTISSAGE_MAX} % de surapprentissage"
            break
        meilleur = admis.loc[admis["validation"].idxmax(), "params"]
        print(f"{nom}, tour {tour} : {len(res)} réglages, {len(admis)} admis ; meilleur admis {admis['validation'].max():.2f} % "
              f"({libelle(meilleur)}) ({round(time.perf_counter() - debut_tour)} s)", flush=True)
        nouvelles = {p: espace["echelles"][p].index(meilleur[p]) for p in positions}
        if nouvelles == positions:
            raison = f"tour {tour} : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle)"
            break
        positions = nouvelles
    historique = pd.concat(lignes, ignore_index=True)
    return historique, raison, round(time.perf_counter() - debut)

resultats = {}
for nom, espace in ESPACES.items():
    historique, raison, duree = optimiser(nom, espace)
    resultats[nom] = {"historique": historique, "raison": raison, "duree": duree}
    admis = historique[historique["admis"]]
    if admis.empty:
        print(f"{nom} : aucun réglage admis ; arrêt : {raison} ({duree} s)")
    else:
        m = admis.loc[admis["validation"].idxmax()]
        print(f"{nom} : {m['validation']:.2f} % ± {m['écart-type des plis']:.2f} ({libelle(m['params'])}) ; arrêt : {raison} ({duree} s)")

Régression logistique, tour 1 : 18 réglages, 18 admis ; meilleur admis 83.93 % (C=3, class_weight=balanced, l1_ratio=0) (16 s)


Régression logistique, tour 2 : 18 réglages, 18 admis ; meilleur admis 83.94 % (C=1, class_weight=balanced, l1_ratio=0.5) (9 s)


Régression logistique, tour 3 : 18 réglages, 18 admis ; meilleur admis 83.94 % (C=1, class_weight=balanced, l1_ratio=0.5) (6 s)


Régression logistique : 83.94 % ± 1.07 (C=1, class_weight=balanced, l1_ratio=0.5) ; arrêt : tour 3 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (31 s)


Arbre de décision, tour 1 : 36 réglages, 33 admis ; meilleur admis 83.86 % (class_weight=balanced, criterion=gini, max_depth=6, min_samples_leaf=5) (2 s)


...

MLP, tour 2 : 9 réglages, 9 admis ; meilleur admis 82.86 % (alpha=0.001, hidden_layer_sizes=(256, 128)) (10 s)


MLP : 82.86 % ± 0.77 (alpha=0.001, hidden_layer_sizes=(256, 128)) ; arrêt : tour 2 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (18 s)


CatBoost, tour 1 : 54 réglages, 48 admis ; meilleur admis 84.45 % (auto_class_weights=Balanced, depth=4, l2_leaf_reg=10, learning_rate=0.03) (182 s)


CatBoost, tour 2 : 54 réglages, 53 admis ; meilleur admis 84.45 % (auto_class_weights=Balanced, depth=4, l2_leaf_reg=10, learning_rate=0.03) (135 s)


CatBoost : 84.45 % ± 0.95 (auto_class_weights=Balanced, depth=4, l2_leaf_reg=10, learning_rate=0.03) ; arrêt : tour 2 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (317 s)


## 7. Synthèse : le meilleur réglage de chaque modèle

Pour chaque modèle, le meilleur réglage admis (surapprentissage de 7 % relatif au plus) sur l'ensemble des tours, avec son score en validation croisée, l'écart-type de ses 5 plis, son score en train et son surapprentissage. Alertes : ⚠️ variance si l'écart-type des plis dépasse 2 points ; ⚠️ aucun réglage admis.

In [8]:
lignes, meilleurs = [], {}
for nom, r in resultats.items():
    h = r["historique"]
    admis = h[h["admis"]]
    if admis.empty:
        ligne = h.loc[h["surapprentissage (% relatif)"].idxmin()]
        alerte = "⚠️ aucun réglage admis (le moins surappris est affiché)"
    else:
        ligne = admis.loc[admis["validation"].idxmax()]
        alerte = f"⚠️ variance ({ligne['écart-type des plis']:.1f} points)" if ligne["écart-type des plis"] > VARIANCE_MAX else "—"
    meilleurs[nom] = ligne["params"]
    lignes.append({"Modèle": nom, "Réglage": libelle(ligne["params"]),
                   f"Validation ({SCORE}, %)": f"{ligne['validation']:.2f} ± {ligne['écart-type des plis']:.2f}",
                   "Train (%)": round(ligne["train"], 2),
                   "Surapprentissage (% relatif)": round(ligne["surapprentissage (% relatif)"], 2),
                   "Alerte": alerte, "Tours": int(h["tour"].max()), "Arrêt": r["raison"], "Durée (s)": r["duree"]})
display(pd.DataFrame(lignes).set_index("Modèle"))

,Réglage,"Validation (familles équilibrées, %)",Train (%),Surapprentissage (% relatif),Alerte,Tours,Arrêt,Durée (s)
Modèle,,,,,,,,
Régression logistique,"C=1, class_weight=balanced, l1_ratio=0.5",83.94 ± 1.07,84.07,0.15,—,3,tour 3 : meilleur réglage stable (à l'intérieu...,31
Arbre de décision,"class_weight=balanced, criterion=entropy, max_...",84.22 ± 0.89,84.37,0.17,—,6,tour 6 : meilleur réglage stable (à l'intérieu...,9
RandomForest,"class_weight=balanced_subsample, max_depth=20,...",84.21 ± 0.85,85.42,1.41,—,4,tour 4 : meilleur réglage stable (à l'intérieu...,1569
MLP,"alpha=0.001, hidden_layer_sizes=(256, 128)",82.86 ± 0.77,83.27,0.49,—,2,tour 2 : meilleur réglage stable (à l'intérieu...,18
CatBoost,"auto_class_weights=Balanced, depth=4, l2_leaf_...",84.45 ± 0.95,86.19,2.01,—,2,tour 2 : meilleur réglage stable (à l'intérieu...,317


## 7 bis. Enregistrement des modèles (`modeles/11b/`)

Chaque modèle est appris sur **tout le train** avec son **meilleur réglage admis** (surapprentissage de 7 % relatif au plus), puis enregistré dans `modeles/11b/11b_<modèle>.pkl`, avec une synthèse `modeles/11b/11b_synthese.json` (données, cible, variables, réglage, scores de validation, durée, versions). Enregistrés avec `cloudpickle` : ils se rechargent avec `pickle.load`, sans redéfinir la fonction du logarithme. Le test de septembre n'est pas lu.

In [9]:
import json, platform
from datetime import datetime
import catboost, sklearn
from joblib.externals import cloudpickle

dossier_modeles = Path.cwd() / "modeles" / "11b"
dossier_modeles.mkdir(parents=True, exist_ok=True)
FICHIERS = {"Régression logistique": "regression_logistique", "Arbre de décision": "arbre_de_decision",
            "RandomForest": "randomforest", "MLP": "mlp", "CatBoost": "catboost"}
synthese = {"scenario": "11b", "date": datetime.now().isoformat(timespec="seconds"), "base": "01e (septembre seul)",
            "source": "data/creditcard_pret_ingestion.csv (niveau 0), sans dpnm, sans ID, sans données personnelles, sans correctifs",
            "cible": "PAY_1 binaire : 0 = sain (-2, -1, 0), 2 = en retard (2 et plus) ; clients dont PAY_1 n'est pas un 1",
            "variables": VARIABLES, "classes": [int(c) for c in CLASSES], "score": SCORE, "surapprentissage_max_pct": SURAPPRENTISSAGE_MAX,
            "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "catboost": catboost.__version__, "pandas": pd.__version__},
            "modeles": {}}
for nom, params in meilleurs.items():
    debut = time.perf_counter()
    modele = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_train, y_train)
    fichier = dossier_modeles / f"11b_{FICHIERS[nom]}.pkl"
    with open(fichier, "wb") as f:
        cloudpickle.dump(modele, f)
    h = resultats[nom]["historique"]
    ligne = h[h["params"].map(lambda p: p == params)].iloc[-1]
    synthese["modeles"][nom] = {"fichier": fichier.name, "reglage": {k.split("__")[1]: (v if isinstance(v, (int, float, str)) or v is None else str(v)) for k, v in params.items()},
                                "validation_pct": round(float(ligne["validation"]), 2), "ecart_type_plis_pct": round(float(ligne["écart-type des plis"]), 2),
                                "train_pct": round(float(ligne["train"]), 2), "surapprentissage_relatif_pct": round(float(ligne["surapprentissage (% relatif)"]), 2),
                                "duree_apprentissage_s": round(time.perf_counter() - debut)}
    print(f"{nom} : enregistré dans {fichier.name}", flush=True)
(dossier_modeles / "11b_synthese.json").write_text(json.dumps(synthese, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Synthèse : {dossier_modeles / '11b_synthese.json'}")

Régression logistique : enregistré dans 11b_regression_logistique.pkl


Arbre de décision : enregistré dans 11b_arbre_de_decision.pkl


RandomForest : enregistré dans 11b_randomforest.pkl


MLP : enregistré dans 11b_mlp.pkl


CatBoost : enregistré dans 11b_catboost.pkl


Synthèse : C:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\lab_ML\correction_codification_1\modeles\11b\11b_synthese.json


## 8. Les modèles réglés, sur tout le train (prédictions hors pli)

Chaque modèle, avec son meilleur réglage, prédit chaque client du train une fois, par le pli qui ne l'a pas appris (`cross_val_predict`, mêmes 5 plis). Les tableaux reprennent ceux des notebooks 01 et 01b, mais sur les 21 049 clients du train au lieu des 4 210 de la validation : les petits groupes (sorties du retard, niveaux de retard élevés) y sont cinq fois plus nombreux.

In [10]:
predictions_hors_pli = {}
for nom, params in meilleurs.items():
    modele = ESPACES[nom]["modele"].set_params(**params)
    predictions_hors_pli[nom] = np.asarray(cross_val_predict(modele, X_train, y_train, cv=cv, n_jobs=ESPACES[nom]["n_jobs"], params=ESPACES[nom].get("fit_params", {}))).ravel().astype(int)
    print(f"{nom} : fait")
predictions_hors_pli["Règle « même codification qu'en août »"] = X_train["PAY_2"].to_numpy()
predictions_hors_pli["Toujours 0"] = np.zeros(len(X_train), dtype=int)

lignes = []
for nom, p in predictions_hors_pli.items():
    b = bonnes_predictions(y_train, p)
    lignes.append({"Modèle": nom, "Bonnes prédictions (%)": round(b["global"], 1),
                   **{f"Famille {NOMS_FAMILLES[f]} (%)": round(b[f], 1) for f in FAMILLES}})
display(pd.DataFrame(lignes).set_index("Modèle"))

Régression logistique : fait
Arbre de décision : fait


RandomForest : fait


MLP : fait


CatBoost : fait


,Bonnes prédictions (%),"Famille sain (-2, -1, 0) (%)",Famille 2 et plus (%)
Modèle,,,
Régression logistique,91.6,94.0,73.8
Arbre de décision,93.5,96.4,72.0
RandomForest,93.3,96.1,72.3
MLP,95.6,99.5,66.2
CatBoost,92.8,95.5,73.4
Règle « même codification qu'en août »,94.5,98.0,68.1
Toujours 0,88.1,100.0,0.0


**Comment lire** : une ligne par modèle réglé, sur les prédictions hors pli du train ; les deux dernières lignes sont les points de comparaison. Mêmes mesures que dans les notebooks 01 et 01b.

In [11]:
famille_aout, famille_sept = famille_pay(X_train["PAY_2"]), famille_pay(y_train)
change = famille_aout != famille_sept
TYPES_CHANGEMENT = {
    "Sortie du retard (2 et plus en août, sain en septembre)": (famille_aout == 2) & (famille_sept != 2),
    "Retard maintenu (2 et plus en août et en septembre ; pas un changement)": (famille_aout == 2) & (famille_sept == 2),
    "Nouveau retard (sain en août, 2 et plus en septembre)": (famille_aout != 2) & (famille_sept == 2),
    "Tous les clients qui changent": change,
}
lignes = []
for type_changement, masque in TYPES_CHANGEMENT.items():
    resultat = {"Type de changement": type_changement, "Clients": int(masque.sum())}
    for nom, p in predictions_hors_pli.items():
        if nom == "Toujours 0":
            continue
        resultat[f"{nom} : retrouvés (%)"] = round((famille_pay(p)[masque] == famille_sept[masque]).mean() * 100, 1)
    lignes.append(resultat)
display(pd.DataFrame(lignes).set_index("Type de changement"))

,Clients,Régression logistique : retrouvés (%),Arbre de décision : retrouvés (%),RandomForest : retrouvés (%),MLP : retrouvés (%),CatBoost : retrouvés (%),Règle « même codification qu'en août » : retrouvés (%)
Type de changement,,,,,,,
"Sortie du retard (2 et plus en août, sain en septembre)",366,1.4,37.4,14.2,78.1,30.6,0.0
Retard maintenu (2 et plus en août et en septembre ; pas un changement),1705,100.0,99.6,99.8,97.0,99.6,100.0
"Nouveau retard (sain en août, 2 et plus en septembre)",799,18.0,13.1,13.5,0.5,17.5,0.0
Tous les clients qui changent,1165,12.8,20.8,13.7,24.9,21.6,0.0


**Comment lire** : comme dans les notebooks 01 et 01b, une ligne par type de changement entre août et septembre, avec la part des clients rangés dans leur vraie famille de septembre ; la règle naïve est à 0 % sur les changements et à 100 % sur les retards maintenus.

## 9. L'importance des variables

**Importance par permutation** : pour chaque variable, ses valeurs sont mélangées entre les clients de la validation, et l'on mesure la baisse du score (en points). Pli par pli (mêmes 5 plis, 5 mélanges par variable), puis moyenne. Le second tableau regroupe les variables par mois : septembre (M : `BILL_AMT1`, `PAY_AMT1`), août (M-1 : `PAY_2`, `BILL_AMT2`, `PAY_AMT2`), … avril (M-5), et le plafond.

In [12]:
from sklearn.inspection import permutation_importance

NOMS_MOIS = {1: "septembre (M)", 2: "août (M-1)", 3: "juillet (M-2)", 4: "juin (M-3)", 5: "mai (M-4)", 6: "avril (M-5)"}
def mois_de(variable):
    return "plafond" if variable == "LIMIT_BAL" else NOMS_MOIS[int(variable[-1])]

importances = {}
for nom, params in meilleurs.items():
    par_pli = []
    for apprentissage, validation in cv.split(X_train, y_train):
        m = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_train.iloc[apprentissage], y_train.iloc[apprentissage])
        r = permutation_importance(m, X_train.iloc[validation], y_train.iloc[validation], scoring=scoreur, n_repeats=5, random_state=42, n_jobs=-1)
        par_pli.append(r.importances_mean * 100)
    importances[nom] = pd.Series(np.mean(par_pli, axis=0), index=X_train.columns)
    print(f"{nom} : fait", flush=True)
table = pd.DataFrame(importances)
table["Moyenne des modèles"] = table.mean(axis=1)
display(Markdown("**Par variable**"))
display(table.sort_values("Moyenne des modèles", ascending=False).round(2))
display(Markdown("**Par mois**"))
display(table.groupby(table.index.map(mois_de)).sum().reindex([NOMS_MOIS[n] for n in range(1, 7)] + ["plafond"]).round(2))

Régression logistique : fait


Arbre de décision : fait


RandomForest : fait


MLP : fait


CatBoost : fait


**Par variable**

,Régression logistique,Arbre de décision,RandomForest,MLP,CatBoost,Moyenne des modèles
PAY_2,31.52,32.39,29.74,31.79,28.47,30.78
PAY_3,1.40,2.05,0.71,-0.03,0.88,1.00
PAY_5,0.29,1.94,0.52,0.19,0.43,0.67
LIMIT_BAL,0.53,1.02,0.23,0.57,0.83,0.64
PAY_AMT2,0.91,0.10,0.15,1.15,0.17,0.50
BILL_AMT2,0.12,0.50,0.11,0.78,0.76,0.45
PAY_4,0.22,0.29,0.05,0.49,0.40,0.29
PAY_6,0.57,0.00,0.34,0.08,0.39,0.28
PAY_AMT3,0.38,0.02,0.07,0.59,0.22,0.26
BILL_AMT1,-0.00,0.00,0.07,0.69,0.46,0.24


**Par mois**

,Régression logistique,Arbre de décision,RandomForest,MLP,CatBoost,Moyenne des modèles
septembre (M),-0.03,0.00,0.22,0.60,0.83,0.32
août (M-1),32.56,32.99,29.99,33.72,29.40,31.73
juillet (M-2),1.93,2.07,0.88,0.83,1.22,1.39
juin (M-3),0.46,0.29,0.28,0.81,0.85,0.54
mai (M-4),0.33,1.94,0.65,0.56,0.55,0.81
avril (M-5),0.70,0.00,0.48,0.26,0.87,0.46
plafond,0.53,1.02,0.23,0.57,0.83,0.64


**Comment lire** : la baisse du score, en points, quand la variable est mélangée. Proche de 0 : le modèle ne s'en sert pas.